# Kinetic-EFIT End-to-End Chain

Drives the full VEST kinetic pipeline over the consolidated `vaft.code.kineticEfit`
API for a single equilibrium slice:

1. **Load** a magnetic-EFIT `gfile` + raw Thomson / charge-exchange diagnostics into an ODS.
2. **`build_kinetic_core_profiles`** — psi_N mapping + `ne`/`Te`/`Ti`/`Vtor` fits + `core_profiles`
   with `pressure_thermal`, `p = e·nₑ·(Tₑ+Tᵢ)` (pure ODS→ODS).
3. **`kinetic_pressure_points`** — the `RPRESS`/`PRESSR`/`SIGPRE`/`FWTPRE` constraint points (raw-6pt).
4. **`run_kinetic_efit`** — kinetic-pressure EFIT with a `PLASMA`-scale sweep (needs `$EFIT` + a base kfile).
5. **`refine_equilibrium`** — CHEASE refinement of the best equilibrium (needs `$CHEASE`).
6. **Compare** the magnetic, kinetic, and CHEASE-refined equilibria.
7. **Derive the Ti/Te coefficient** from the database — scan the shots that carry both
   diagnostics and fit `Ti = α·Te` on their stored kinetic profiles.
8. **Thomson-only shots** — the statistical `Ti = 0.17·Te` fallback (no ion Doppler data needed;
   validated to a few % of the two-diagnostic pressure).
9. **TS+IDS EFIT vs TS-only EFIT** — rerun the kinetic EFIT from the statistical pressure and
   compare the two reconstructed equilibria.

This notebook runs on the **physically-paired VEST shot 48224 @ 300 ms** dataset packaged
under `vaft/data/kineticEfit/` (equilibrium `g048224.00300`, Thomson `NeTe_48224.mat`,
ion Doppler `IDS_48224.mat` — all the same shot and time). Each external-code stage
degrades gracefully: if `$EFIT` / a base kfile / `$CHEASE` is unset the notebook prints a
clear `skipped: …` message and the remaining cells stay executable (on the VEST lab machine
the EFIT binary and the filedb base kfile resolve automatically).

Override `VAFT_KINETIC_DATA_ROOT` to point at another paired dataset; set `$EFIT` +
`$VAFT_KINETIC_BASE_KFILE` and `$CHEASE` to run the kinetic-EFIT and CHEASE stages.

In [ ]:
import os
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np

import vaft
from vaft.data.resources import data_path
from vaft.data import read_geqdsk
from vaft.code.kineticEfit import (
    build_kinetic_core_profiles,
    kinetic_pressure_points,
    KineticEFITConfig,
    prepare_kinetic_efit_inputs,
    run_kinetic_efit,
)
from vaft.code.chease import CHEASEConfig, find_chease_executable, refine_equilibrium

vaft.apply_omfit_compat_patches()
from omfit_classes.omfit_eqdsk import OMFITgeqdsk

# --- physically-paired VEST shot 48224 @ 300 ms (vaft/data/kineticEfit/) ---
SHOT    = 48224
TIME_MS = 300.0
DATA_ROOT = Path(os.environ.get("VAFT_KINETIC_DATA_ROOT", str(data_path() / "kineticEfit")))
GFILE    = DATA_ROOT / f"g0{SHOT}.{int(TIME_MS):05d}"
TS_MAT   = DATA_ROOT / f"NeTe_{SHOT}.mat"
ION_MAT  = DATA_ROOT / f"IDS_{SHOT}.mat"
ION_OPT  = "ids"
TIME_TOL_MS = 3.0   # small: gfile / Thomson / ion diagnostics are all this same 300 ms slice
WORKROOT = Path(tempfile.mkdtemp(prefix="vaft-kinetic-"))

# Binaries / base kfile resolved from env vars, falling back to the VEST lab
# defaults when they exist on this machine (graceful skip otherwise).
_vest_efit = Path("/home/user1/work/efit-new2/build/efit/efit")   # VEST lab build
EFIT_EXE   = os.environ.get("EFIT") or (str(_vest_efit) if _vest_efit.exists() else None)
# Base magnetic kfile the pressure block patches; default to the VEST filedb path.
_filedb_kfile = Path(f"/srv/vest.filedb/public/{SHOT}/efit/kfile/k0{SHOT}.{int(TIME_MS):05d}")
BASE_KFILE = os.environ.get("VAFT_KINETIC_BASE_KFILE") or (str(_filedb_kfile) if _filedb_kfile.exists() else None)
_chease_found = find_chease_executable()
CHEASE_EXE = str(_chease_found) if _chease_found else None

print(f"Data root  : {DATA_ROOT}")
print(f"Slice      : shot {SHOT} @ {TIME_MS:.0f} ms  (eq {GFILE.name} + TS/ion {SHOT})")
print(f"Work dir   : {WORKROOT}")
print(f"EFIT exe   : {EFIT_EXE or '(unset -> kinetic EFIT skipped)'}")
print(f"base kfile : {BASE_KFILE or '(unset -> kinetic EFIT skipped)'}")
print(f"CHEASE exe : {CHEASE_EXE or '(unset -> CHEASE refine skipped)'}")

## 1. Load The Magnetic Equilibrium And Raw Kinetic Diagnostics

The magnetic-EFIT `gfile` seeds the ODS `equilibrium`; Thomson scattering supplies
`nₑ`/`Tₑ` and the ion Doppler (IDS) supplies `Tᵢ`/`Vtor`. I/O only — no profile physics yet.

In [ ]:
geq = OMFITgeqdsk(str(GFILE))
geq["fluxSurfaces"].load()

ods = geq.to_omas()
ods["equilibrium.ids_properties.homogeneous_time"] = 1
vaft.machine_mapping.dataset_description(
    ods, source=SHOT,
    options={"source_type": "shot",
             "description": f"VEST kinetic chain shot {SHOT} @ {TIME_MS:.0f} ms (TS + ion Doppler)"},
)

# electrons: Thomson scattering; ions: IDS -> charge_exchange (same shot 48224)
vaft.machine_mapping.thomson_scattering(ods, SHOT, str(TS_MAT))
vaft.machine_mapping.charge_exchange(ods, shotnumber=SHOT, options=ION_OPT, mat_file=str(ION_MAT))

print(f"gfile        : {GFILE.name}")
print(f"Ip (magnetic): {float(geq['CURRENT']):.6e} A")
print(f"B0           : {float(geq['BCENTR']):.4f} T")
print(f"TS channels  : {len(ods['thomson_scattering.channel'])}")
print(f"CX channels  : {len(ods['charge_exchange.channel'])}")

## 2. Build The Kinetic Core Profiles

`build_kinetic_core_profiles` is a pure ODS→ODS orchestration of the `vaft.process.profile`
chain: it maps TS and CX onto the equilibrium `psi_N`, fits `nₑ`/`Tₑ` (TS) and `Tᵢ`/`Vtor`
(ion diagnostic), and writes `core_profiles.profiles_1d[0]` including `pressure_thermal`
(`p = e·nₑ·(Tₑ+Tᵢ)`, `n_i ≈ n_e`). `time_tolerance_ms` is widened here because the demo
pairs a 320 ms equilibrium with 300 ms diagnostics.

In [ ]:
ods = build_kinetic_core_profiles(
    ods, geq, TIME_MS,
    te_mode="polynomial",
    ne_mode="free_exponential",
    ti_mode="polynomial",
    vtor_mode="polynomial",
    ion_index=0,
    time_tolerance_ms=TIME_TOL_MS,
)

cp = "core_profiles.profiles_1d.0"
rho  = np.asarray(ods[f"{cp}.grid.rho_tor_norm"])
ne   = np.asarray(ods[f"{cp}.electrons.density"])
te   = np.asarray(ods[f"{cp}.electrons.temperature"])
ti   = np.asarray(ods[f"{cp}.ion.0.temperature"])
vtor = np.asarray(ods[f"{cp}.ion.0.velocity.toroidal"])
pth  = np.asarray(ods[f"{cp}.pressure_thermal"])

print(f"grid points : {rho.size}")
print(f"ne0         : {ne[0]:.3e} m^-3")
print(f"Te0         : {te[0]:.1f} eV")
print(f"Ti0         : {ti[0]:.1f} eV")
print(f"Vtor0       : {vtor[0]:.3e} m/s")
print(f"p_th0       : {pth[0]:.1f} Pa")

In [ ]:
# raw measured points WITH error bars, read straight from the diagnostic channels
# and mapped psi_N -> rho_tor on the equilibrium grid.
psi_grid = np.asarray(ods[f"{cp}.grid.psi"], dtype=float)
psin_grid = (psi_grid - psi_grid[0]) / (psi_grid[-1] - psi_grid[0])
def _to_rhotor(psiN):
    return np.interp(np.clip(np.asarray(psiN, dtype=float), 0, 1), psin_grid, rho)

# Thomson channels -> (rho_tor, value, error) at this time
ts_psiN = np.asarray(vaft.process.equilibrium_mapping_thomson_scattering(ods, geq), dtype=float)
it = int(np.argmin(np.abs(np.asarray(ods["thomson_scattering.time"], dtype=float) - TIME_MS / 1e3)))
n_ts = len(ods["thomson_scattering.channel"])
def _ts(field):
    v = np.array([float(np.asarray(ods[f"thomson_scattering.channel.{i}.{field}.data"], dtype=float)[it]) for i in range(n_ts)])
    e = np.array([float(np.asarray(ods[f"thomson_scattering.channel.{i}.{field}.data_error_upper"], dtype=float)[it]) for i in range(n_ts)])
    return v, e
ne_v, ne_e = _ts("n_e"); te_v, te_e = _ts("t_e")
ts_rho = _to_rhotor(ts_psiN)

# charge_exchange channels -> (rho_tor, value, error) at this time
cx_psiN = np.asarray(vaft.process.equilibrium_mapping_charge_exchange(ods, geq), dtype=float)
jt = int(np.argmin(np.abs(np.asarray(ods["charge_exchange.time"], dtype=float) - TIME_MS / 1e3)))
n_cx = len(ods["charge_exchange.channel"])
def _cx(field):
    v, e = [], []
    for i in range(n_cx):
        ion = ods[f"charge_exchange.channel.{i}.ion.0.{field}"]
        v.append(float(np.asarray(ion["data"], dtype=float)[jt]))
        try:
            e.append(float(np.asarray(ion["data_error_upper"], dtype=float)[jt]))
        except Exception:
            e.append(np.nan)
    return np.array(v), np.array(e)
ti_v, ti_e = _cx("t_i"); vtor_v, vtor_e = _cx("velocity_tor")
cx_rho = _to_rhotor(cx_psiN)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panels = [
    (ne,   ts_rho, ne_v,   ne_e,   r"$n_e$ [m$^{-3}$]",   "tab:blue"),
    (te,   ts_rho, te_v,   te_e,   r"$T_e$ [eV]",         "tab:red"),
    (ti,   cx_rho, ti_v,   ti_e,   r"$T_i$ [eV]",         "tab:orange"),
    (vtor, cx_rho, vtor_v, vtor_e, r"$V_{tor}$ [m/s]",    "tab:green"),
    (pth,  None,   None,   None,   r"$p_{thermal}$ [Pa]", "tab:purple"),
]
for ax, (y, rx, rv, re, ylabel, color) in zip(axes.flat, panels):
    ax.plot(rho, y, color=color, lw=2, label="fit")
    if rx is not None:
        m = np.isfinite(rx) & np.isfinite(rv)
        yerr = np.where(np.isfinite(re[m]), re[m], 0.0)
        ax.errorbar(rx[m], rv[m], yerr=yerr, fmt="o", color=color, mfc="white",
                    ms=5, capsize=3, elinewidth=1, label="raw ± σ")
        ax.legend(fontsize=8)
    ax.set_xlabel(r"$\rho_{tor,N}$")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
axes.flat[-1].axis("off")
fig.suptitle(f"Kinetic core profiles @ {TIME_MS:.0f} ms  (line = fit, points = raw ± σ)", fontsize=13)
fig.tight_layout()
plt.show()

## 3. Build The Kinetic-Pressure Constraint Points

`kinetic_pressure_points` turns the fitted profiles into the EFIT pressure constraint.
With `encoding='raw6'` it emits the 5 Thomson points at their real major radius `R>0` plus a
6th `psi_N=1, p=0` edge anchor (`RPRESS=-1.0`). Each carries `PRESSR = e·nₑ·(Tₑ+Tᵢ)` and the
propagated `SIGPRE`.

In [ ]:
points = kinetic_pressure_points(ods, TIME_MS, geq=geq, encoding="raw6")

print(f"encoding : raw6  ({len(points.rpress)} points)")
print(f"{'RPRESS':>10}{'PRESSR[Pa]':>14}{'SIGPRE[Pa]':>14}{'FWTPRE':>10}")
for i in range(len(points.rpress)):
    print(f"{points.rpress[i]:>10.4f}{points.pressr[i]:>14.3f}"
          f"{points.sigpre[i]:>14.3f}{points.fwtpre[i]:>10.3f}")

rp = np.asarray(points.rpress, dtype=float)
pr = np.asarray(points.pressr, dtype=float)
sg = np.asarray(points.sigpre, dtype=float)
real = rp > 0.0        # the 5 Thomson points at their real major radius R
anchor = ~real         # the synthetic RPRESS=-1.0 (psi_N=1) edge anchor, p=0

# place the psi_N=1 anchor at the outboard separatrix major radius so it shows on R
try:
    R_sep = float(np.max(np.asarray(geq["RBBBS"], dtype=float)))
except Exception:
    R_sep = float(rp[real].max())

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.errorbar(rp[real], pr[real], yerr=sg[real], fmt="o", color="tab:purple",
            capsize=4, label=f"{int(real.sum())} Thomson points")
ax.plot([R_sep] * int(anchor.sum()), pr[anchor], "X", color="crimson", ms=13,
        label=r"edge anchor ($\psi_N=1,\ p=0$)")
ax.axhline(0.0, color="grey", lw=0.8, ls=":")
ax.set_xlabel("R [m]")
ax.set_ylabel(r"$p = e\,n_e\,(T_e + T_i)$  [Pa]")
ax.set_title("Kinetic-pressure constraint points (raw6)")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 4. Run The Kinetic-Pressure EFIT

`run_kinetic_efit` injects the pressure constraint into the base magnetic kfile, runs EFIT
across the `PLASMA`-scale sweep, and keeps the largest scale that converges. It needs `$EFIT`
and a base magnetic kfile (`$VAFT_KINETIC_BASE_KFILE`) — the packaged sample has none, so this
cell prints a clear `skipped:` line unless both are provided on a real slice.

In [ ]:
efit_result = None
if not EFIT_EXE or not BASE_KFILE:
    print("skipped: set $EFIT and $VAFT_KINETIC_BASE_KFILE (a magnetic kfile) to run the "
          "kinetic-pressure EFIT. The packaged demo sample ships no base kfile.")
else:
    try:
        efit_config = KineticEFITConfig(
            executable=EFIT_EXE,
            workdir=str(WORKROOT / "kinetic_efit"),
            shot=SHOT,
            time_ms=TIME_MS,
            encoding="raw6",
            base_kfile=BASE_KFILE,
            timeout=600,
        )
        efit_inputs = prepare_kinetic_efit_inputs(ods, geq, efit_config)
        efit_result = run_kinetic_efit(efit_inputs, efit_config)
        if efit_result.status == "skipped":
            print(f"skipped: {efit_result.reason}")
        elif efit_result.converged:
            print(f"converged at PLASMA scale = {efit_result.scale}")
            print(f"gfile : {efit_result.gfile}")
        else:
            print(f"did not converge (status={efit_result.status}): {efit_result.reason}")
    except Exception as exc:
        print(f"skipped: kinetic EFIT raised {type(exc).__name__}: {exc}")

## 5. Refine The Equilibrium With CHEASE

`refine_equilibrium` runs the CHEASE inverse solver (pure-python jsk95 parity) on the best
available equilibrium — the kinetic-EFIT `gfile` if that step converged, otherwise the magnetic
`gfile`. CHEASE is discovered from `$CHEASEHOME/bin/chease`, with `CHEASE` and
`CHEASE_EXEC_DIR` retained for compatibility; the cell prints a clear
`skipped:` line when no executable is found.

In [ ]:
if efit_result is not None and getattr(efit_result, "gfile", None):
    chease_source, src_label = efit_result.gfile, "kinetic-EFIT gfile"
else:
    chease_source, src_label = GFILE, "magnetic gfile (kinetic EFIT unavailable)"

chease_result = None
if not CHEASE_EXE:
    print("skipped: set $CHEASEHOME to a root containing bin/chease.")
else:
    try:
        chease_config = CHEASEConfig(
            executable=str(CHEASE_EXE),
            workdir=str(WORKROOT / "chease"),
            create_plot=False,
            cleanup=False,
            timeout=350,
        )
        print(f"refining {src_label}: {Path(str(chease_source)).name}")
        chease_result = refine_equilibrium(str(chease_source), chease_config)
        if chease_result.ok and chease_result.refined_geqdsk:
            print(f"refined gfile : {chease_result.refined_geqdsk}")
            for key, value in (chease_result.comparison or {}).items():
                print(f"  {key}: {value:.6g}")
        else:
            print("CHEASE produced no refined GEQDSK in this environment.")
            tail = (chease_result.stderr or "").strip().splitlines()[-8:]
            if tail:
                print("\n".join(tail))
    except Exception as exc:
        print(f"skipped: CHEASE raised {type(exc).__name__}: {exc}")

## 6. Compare Magnetic vs Kinetic vs CHEASE Equilibria

Overlay `q(ψ_N)`, `p(ψ_N)`, and the boundary for every equilibrium that is actually available.
With no binaries only the magnetic equilibrium is shown; `$EFIT`+base-kfile overlays the kinetic
reconstruction, and `$CHEASE` overlays the refined result.

In [ ]:
def _try_read(path):
    try:
        return read_geqdsk(str(path)) if path else None
    except Exception:
        return None


eqs = [("Magnetic EFIT", GFILE, "tab:blue", "-")]
if efit_result is not None and getattr(efit_result, "gfile", None):
    eqs.append(("Kinetic EFIT", efit_result.gfile, "tab:red", "-"))
if chease_result is not None and getattr(chease_result, "refined_geqdsk", None):
    eqs.append(("CHEASE refined", chease_result.refined_geqdsk, "tab:green", "--"))

loaded = [(label, _try_read(path), color, ls) for label, path, color, ls in eqs]
loaded = [t for t in loaded if t[1] is not None]

fig, (ax_q, ax_p, ax_b) = plt.subplots(1, 3, figsize=(16, 5))
for label, g, color, ls in loaded:
    x = np.linspace(0.0, 1.0, int(g["NW"]))
    ax_q.plot(x, np.asarray(g["QPSI"], dtype=float), color=color, ls=ls, lw=1.8, label=label)
    ax_p.plot(x, np.asarray(g["PRES"], dtype=float), color=color, ls=ls, lw=1.8, label=label)
    rb = np.asarray(g.get("RBBBS", []), dtype=float)
    zb = np.asarray(g.get("ZBBBS", []), dtype=float)
    if rb.size and zb.size:
        n = min(rb.size, zb.size)
        ax_b.plot(rb[:n], zb[:n], color=color, ls=ls, lw=1.8, label=label)

ax_q.set_title("Safety factor q"); ax_q.set_xlabel(r"$\psi_N$"); ax_q.grid(True, alpha=0.25); ax_q.legend()
ax_p.set_title("Pressure [Pa]"); ax_p.set_xlabel(r"$\psi_N$"); ax_p.grid(True, alpha=0.25); ax_p.legend()
ax_b.set_title("Boundary"); ax_b.set_xlabel("R [m]"); ax_b.set_ylabel("Z [m]")
ax_b.set_aspect("equal", adjustable="box"); ax_b.grid(True, alpha=0.25); ax_b.legend()
fig.suptitle(f"Equilibrium comparison @ {TIME_MS:.0f} ms", fontsize=13)
fig.tight_layout()
plt.show()

print(f"Equilibria compared: {[label for label, *_ in loaded]}")
if len(loaded) == 1:
    print("Only the magnetic equilibrium is available - set $EFIT + $VAFT_KINETIC_BASE_KFILE "
          "and $CHEASE to populate the kinetic and refined overlays.")

print("\nOne-call equivalent of this whole notebook:")
print("  from vaft.code.kineticEfit import run_kinetic_chain")
print("  out = run_kinetic_chain(ods, geq, TIME_MS, efit_config=efit_config,")
print("                          chease_config=chease_config)")

## 7. Deriving The Ti/Te Coefficient From The Database

Where does `TI_TE_RATIO_VEST` come from? This section reproduces the derivation
end-to-end from the VEST HSDS database, so the coefficient can be **re-derived and
updated** as more two-diagnostic shots are processed.

**Finding the data.** `vaft.database.exist_shot(data_filter='cx')` lists the shots the
corrective pipeline processed for charge exchange (ion Doppler); `'ts'` and `'cp'` list
the Thomson and kinetic-core_profiles registries. Loading a shot with
`vaft.database.load(shot)` gives the stored ODS, and every `core_profiles.profiles_1d`
slice that carries a *measured* ion temperature (i.e. `ion[0].temperature` present and
not simply a copy of `Te`) is one sample for the fit.

**The estimator.** The coefficient multiplies the **fitted** electron profile — the same
`core_profiles` curves the spline pressure encoding consumes — so it is derived
fitted-vs-fitted. Per slice we take the *pressure-matching* coefficient, the α that
preserves the kinetic pressure when $T_i$ is replaced by $\alpha T_e$:

$$\min_\alpha \sum_j \left[\,e n_{e,j} T_{i,j} - \alpha\, e n_{e,j} T_{e,j}\right]^2
\;\;\Longrightarrow\;\;
\alpha = \frac{\sum_j n_{e,j}^2 T_{e,j} T_{i,j}}{\sum_j n_{e,j}^2 T_{e,j}^2}
\qquad (\psi_N \le 1)$$

`vaft.process.profile.fit_ti_te_ratio` provides the complementary estimator (an
effective-variance through-origin regression with errors in both variables) for
point-wise data; it is evaluated alongside for comparison.

The adopted constants (`TI_TE_RATIO_VEST = 0.17`, `TI_TE_RATIO_VEST_SIGMA = 0.08`) come
from the 9-slice reference set (shots 48224/48226/48233 @ 299–301 ms: slice mean 0.190,
median 0.155, std 0.080). The cell below re-runs the same procedure over whatever the
database currently holds and compares. It needs HSDS access (`hsconfigure`) and takes
~30 s per shot; without it, it prints `skipped:` and the rest of the notebook still runs.

In [ ]:
from vaft.process.profile import (
    TI_TE_RATIO_VEST, TI_TE_RATIO_VEST_SIGMA, fit_ti_te_ratio,
)

MAX_SHOTS = None        # limit the scan (None = every shot in the CX registry)
EQE = 1.602176634e-19


def _slice_profiles(ods, i):
    """(psi_N, Te, ne, Ti) of core_profiles.profiles_1d[i], or None.

    Returns None when the slice has no measured ion-temperature metadata. This
    excludes both the legacy Ti = Te fallback and the statistical Ti = alpha*Te
    fallback, preventing modeled slices from feeding back into alpha itself.
    """
    b = f"core_profiles.profiles_1d.{i}"
    try:
        te = np.asarray(ods[f"{b}.electrons.temperature"], dtype=float)
        ne = np.asarray(ods[f"{b}.electrons.density"], dtype=float)
        ti = np.asarray(ods[f"{b}.ion.0.temperature"], dtype=float)
        ti_measured = np.asarray(
            ods[f"{b}.ion.0.temperature_fit.measured"], dtype=float
        ).reshape(-1)
    except Exception:
        return None
    if ti_measured.size == 0 or not np.any(np.isfinite(ti_measured)):
        return None
    if te.shape != ti.shape or te.shape != ne.shape:
        return None
    try:                                    # psi_N of the stored grid
        psi = np.asarray(ods[f"{b}.grid.psi"], dtype=float)
        psin = (psi - psi[0]) / (psi[-1] - psi[0])
    except Exception:
        psin = np.linspace(0.0, 1.0, te.size)
    m = ((psin <= 1.0) & (te > 0)
         & np.isfinite(te) & np.isfinite(ti) & np.isfinite(ne))
    if m.sum() < 5:
        return None
    return psin[m], te[m], ne[m], ti[m]


samples = []          # (shot, t_ms, alpha_pressure_matched, alpha_fit_ti_te_ratio, psi_N, Ti/Te)
try:
    cx_table = vaft.database.exist_shot(data_filter="cx")
    cx_shots = sorted(int(s) for s in cx_table["Shot Number"])
    if MAX_SHOTS:
        cx_shots = cx_shots[:MAX_SHOTS]
    print(f"CX registry: {len(cx_shots)} shot(s) -> {cx_shots}")

    for shot_i in cx_shots:
        try:
            ods_i = vaft.database.load(shot_i)
        except Exception as exc:
            print(f"  {shot_i}: load failed ({type(exc).__name__})")
            continue
        if "core_profiles" not in ods_i:
            print(f"  {shot_i}: no core_profiles stored -- skipped")
            continue
        found = 0
        for i in range(len(ods_i["core_profiles.profiles_1d"])):
            prof = _slice_profiles(ods_i, i)
            if prof is None:
                continue
            psin_i, te_i, ne_i, ti_i = prof
            # primary: pressure-matching coefficient on the fitted profiles
            a_pm = float(np.sum(ne_i**2 * te_i * ti_i) / np.sum(ne_i**2 * te_i**2))
            # complementary: the vaft estimator on the same (Te, Ti) pairs
            a_fit = fit_ti_te_ratio(te_i, ti_i)["alpha"]
            t_ms_i = float(ods_i[f"core_profiles.profiles_1d.{i}.time"]) * 1e3
            samples.append((shot_i, t_ms_i, a_pm, a_fit, psin_i, ti_i / te_i))
            found += 1
        print(f"  {shot_i}: {found} slice(s) with a measured ion temperature")
except Exception as exc:
    print(f"skipped: database scan unavailable ({type(exc).__name__}: {exc}). "
          "Run `hsconfigure` for HSDS access.")

if samples:
    a_pm_all = np.array([s[2] for s in samples])
    a_fit_all = np.array([s[3] for s in samples])
    print(f"\n{'shot':>7}{'t [ms]':>9}{'alpha_pm':>10}{'alpha_fit':>11}")
    for shot_i, t_ms_i, a_pm, a_fit, *_ in samples:
        print(f"{shot_i:>7}{t_ms_i:>9.1f}{a_pm:>10.4f}{a_fit:>11.4f}")
    print(f"\nN = {a_pm_all.size} slices | pressure-matched: mean {a_pm_all.mean():.4f}  "
          f"median {np.median(a_pm_all):.4f}  std {a_pm_all.std(ddof=1):.4f}")
    print(f"{'':>21}fit_ti_te_ratio : mean {a_fit_all.mean():.4f}  "
          f"std {a_fit_all.std(ddof=1):.4f}")
    rng = np.random.default_rng(0)
    boots = rng.choice(a_pm_all, size=(5000, a_pm_all.size), replace=True).mean(axis=1)
    lo_b, hi_b = np.percentile(boots, [2.5, 97.5])
    print(f"bootstrap 95% CI of the slice mean: [{lo_b:.4f}, {hi_b:.4f}]")
    print(f"\nadopted in vaft: TI_TE_RATIO_VEST = {TI_TE_RATIO_VEST} "
          f"+/- {TI_TE_RATIO_VEST_SIGMA}  "
          f"({'consistent' if lo_b <= TI_TE_RATIO_VEST <= hi_b else 'OUTSIDE the CI -- revisit'})")

    fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.6))
    # (a) the fitted Ti/Te profiles every slice contributes -- shown only over
    # the psi_N region the diagnostics actually cover (Thomson channels sit
    # inside psi_N ~ 0.4); further out the curves are pure fit extrapolation.
    PSIN_SHOW = 0.6
    for shot_i, t_ms_i, a_pm, _a, psin_i, ratio_i in samples:
        m = psin_i <= PSIN_SHOW
        axa.plot(psin_i[m], ratio_i[m], lw=1.6, label=f"{shot_i} @ {t_ms_i:.0f} ms")
    axa.axhline(TI_TE_RATIO_VEST, color="k", lw=1.9,
                label=f"adopted α = {TI_TE_RATIO_VEST}")
    axa.axhspan(TI_TE_RATIO_VEST - TI_TE_RATIO_VEST_SIGMA,
                TI_TE_RATIO_VEST + TI_TE_RATIO_VEST_SIGMA, color="k", alpha=0.12,
                label=f"±σ = {TI_TE_RATIO_VEST_SIGMA}")
    axa.set_xlabel(r"$\psi_N$"); axa.set_ylabel(r"$T_i/T_e$ (fitted profiles)")
    axa.set_xlim(0.0, PSIN_SHOW)
    axa.set_ylim(0, max(0.75, float(np.max(
        [r[x <= PSIN_SHOW].max() for *_z, x, r in samples])) * 1.05))
    axa.grid(alpha=0.3); axa.legend(fontsize=7, ncol=2)
    axa.set_title(f"fitted Ti/Te profile of each contributing slice "
                  f"($\\psi_N \\leq$ {PSIN_SHOW}, the measured region)")

    # (b) per-slice alpha vs the adopted constant
    idx = np.arange(len(samples))
    axb.bar(idx - 0.2, a_pm_all, width=0.4, color="tab:purple",
            label="pressure-matched")
    axb.bar(idx + 0.2, a_fit_all, width=0.4, color="tab:purple", alpha=0.45,
            label="fit_ti_te_ratio")
    axb.axhline(TI_TE_RATIO_VEST, color="k", lw=1.7)
    axb.axhspan(TI_TE_RATIO_VEST - TI_TE_RATIO_VEST_SIGMA,
                TI_TE_RATIO_VEST + TI_TE_RATIO_VEST_SIGMA, color="k", alpha=0.12)
    axb.set_xticks(idx)
    axb.set_xticklabels([f"{s[0]}\n{s[1]:.0f} ms" for s in samples], fontsize=7)
    axb.set_ylabel("per-slice α"); axb.grid(alpha=0.3, axis="y"); axb.legend(fontsize=8)
    axb.set_title(f"per-slice α (mean {a_pm_all.mean():.3f}, std {a_pm_all.std(ddof=1):.3f})")
    fig.suptitle("Deriving the Ti/Te coefficient from the stored kinetic profiles", fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print("no two-diagnostic slices found -- the adopted constant is used as-is.")

## 8. Thomson-Only Shots — Statistical Ti/Te Fallback

Most VEST shots carry Thomson scattering but **no ion Doppler (IDS/charge_exchange)** data.
For those, the kinetic chain falls back to a **statistical ion temperature**

$$T_i = \alpha\,T_e,\qquad \alpha = \texttt{TI\_TE\_RATIO\_VEST} = 0.17 \pm 0.08,$$

so the kinetic pressure becomes $p = e\,n_e\,(1+\alpha)\,T_e$ with the ratio uncertainty
propagated into `SIGPRE`: $\sigma_{T_i} = \sqrt{(\alpha\,\sigma_{T_e})^2 + (\sigma_\alpha\,T_e)^2}$.

**Provenance** — $\alpha$ is derived from the 9 slices that carry *both* profiles
(shots 48224/48226/48233 @ 299–301 ms) using the **fitted 129-pt core_profiles curves**
(the same profiles the spline pressure encoding consumes): per slice, the pressure-matching
coefficient $\alpha = \sum n_e^2 T_e T_i \,/\, \sum n_e^2 T_e^2$ — the value that preserves
the kinetic pressure when $T_i$ is replaced by $\alpha T_e$ (slice mean 0.19, median 0.155,
std 0.08). An independent raw-point pairing (measured TS $T_e$ vs the weighted CX $T_i$ fit,
effective-variance through-origin regression) cross-checks at 0.170 ± 0.010, robust to the
mapping equilibrium, fit degree and pairing direction (0.163–0.171). Validated against the
true two-diagnostic pressure: **−0.1 % ± 5.6 % (max 15 %)** — the legacy `Ti = Te` assumption
overestimates by **+71 %**. Known trend: the early 299 ms slices sit high (~0.28) vs the
300/301 ms cluster (~0.14). **Section 7 above re-derives $\alpha$ from the database**; estimator:
`vaft.process.profile.fit_ti_te_ratio`.

**Control** via `ti_te_ratio` on `build_kinetic_core_profiles`, `kinetic_pressure_points`
and `KineticEFITConfig`:

| `ti_te_ratio` | behaviour when ion data is missing |
|---|---|
| `'auto'` *(default)* | `Ti = 0.17·Te`, kinetic `pressure_thermal` written, `[INFO]` logged |
| float (e.g. `0.3`) | forced coefficient (pair with `ti_te_ratio_sigma`) |
| `None` | strict legacy — raise (points) / `Ti = Te`, **no** pressure (profiles) |

Shots **with** ion data are never affected — the measured Ti always takes precedence.
The cell below rebuilds this same slice *without* the ion file and compares the fallback
constraint points against the real two-diagnostic ones from section 3.

In [ ]:
from vaft.process.profile import TI_TE_RATIO_VEST, TI_TE_RATIO_VEST_SIGMA

# --- rebuild the SAME slice with ONLY Thomson scattering (no ion Doppler) ---
ods_ts = geq.to_omas()
ods_ts["equilibrium.ids_properties.homogeneous_time"] = 1
vaft.machine_mapping.thomson_scattering(ods_ts, SHOT, str(TS_MAT))

# identical call to section 2 -- the [INFO] line marks the statistical fallback
ods_ts = build_kinetic_core_profiles(ods_ts, geq, TIME_MS, time_tolerance_ms=TIME_TOL_MS)

cpt = "core_profiles.profiles_1d.0"
te_ts  = np.asarray(ods_ts[f"{cpt}.electrons.temperature"])
ti_ts  = np.asarray(ods_ts[f"{cpt}.ion.0.temperature"])
pth_ts = np.asarray(ods_ts[f"{cpt}.pressure_thermal"])
print(f"\nratio          : Ti/Te = {TI_TE_RATIO_VEST} +/- {TI_TE_RATIO_VEST_SIGMA}")
print(f"Ti0 (fallback) : {ti_ts[0]:.1f} eV   (real IDS fit gave {ti[0]:.1f} eV)")
print(f"p_th0          : {pth_ts[0]:.1f} Pa  vs two-diagnostic {pth[0]:.1f} Pa "
      f"({(pth_ts[0] / pth[0] - 1) * 100:+.1f}%)")

# --- raw6 constraint points: fallback vs the real two-diagnostic ones (section 3) ---
pts_ts = kinetic_pressure_points(ods_ts, TIME_MS, geq=geq, encoding="raw6")

rp_ts = np.asarray(pts_ts.rpress, dtype=float)
pr_ts = np.asarray(pts_ts.pressr, dtype=float)
sg_ts = np.asarray(pts_ts.sigpre, dtype=float)
real_ts = rp_ts > 0.0
# for contrast: what the removed legacy assumption Ti=Te would have fed EFIT
p_e = pr_ts[real_ts] / (1.0 + TI_TE_RATIO_VEST)      # electron pressure e*ne*Te
p_tite = 2.0 * p_e                                    # legacy p = e*ne*2*Te

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.errorbar(rp[real], pr[real], yerr=sg[real], fmt="o", color="tab:purple", capsize=4,
            ms=7, label="TS + ion Doppler (real Ti)")
ax.errorbar(rp_ts[real_ts] + 0.004, pr_ts[real_ts], yerr=sg_ts[real_ts], fmt="s",
            color="tab:orange", capsize=4, ms=7, mfc="white",
            label=rf"TS-only fallback  $T_i={TI_TE_RATIO_VEST}\,T_e$")
ax.plot(rp_ts[real_ts], p_tite, "^", color="tab:red", ms=8, alpha=0.7,
        label=r"legacy $T_i=T_e$ (removed default)")
ax.plot(R_sep, 0.0, "X", color="crimson", ms=13, label=r"edge anchor ($\psi_N=1$)")
ax.axhline(0.0, color="grey", lw=0.8, ls=":")
ax.set_xlabel("R [m]")
ax.set_ylabel(r"$p$  [Pa]")
ax.set_title("Kinetic-pressure constraint: real ion data vs TS-only statistical fallback")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

dev = (pr_ts[real_ts] / pr[real] - 1) * 100
print(f"fallback vs real per point: {np.array2string(dev, precision=1)} %  "
      f"(fallback SIGPRE covers the deviation)")

# strict mode: Thomson-only + ti_te_ratio=None keeps the legacy hard failure
try:
    kinetic_pressure_points(ods_ts, TIME_MS, geq=geq, encoding="raw6", ti_te_ratio=None)
except Exception as exc:
    print(f"strict mode (ti_te_ratio=None) raises as before -> {type(exc).__name__}: {exc}")

print("\nRe-derive the coefficient when new two-diagnostic shots appear:")
print("  from vaft.process.profile import fit_ti_te_ratio")
print("  fit_ti_te_ratio(te_pairs, ti_pairs, te_std, ti_std)  # -> {'alpha', 'alpha_se', ...}")

## 9. Compare The TS+IDS EFIT vs The TS-Only Approximated EFIT

The real question for TS-only shots: **how different is the reconstructed equilibrium?**
This section runs the kinetic-pressure EFIT a second time from the *same* base magnetic kfile
and `PLASMA`-scale sweep, but with the constraint points built from the TS-only statistical
pressure (`ods_ts`, section 7) instead of the real two-diagnostic pressure (`ods`, section 4),
then compares the two solutions: `p(ψ_N)`, `q(ψ_N)`, the boundary, and the scalar figures of
merit (converged scale, Ip, p₀, q₀, q₉₅).

Needs `$EFIT` + a base magnetic kfile like section 4 (on the VEST lab machine both resolve
automatically); otherwise it prints `skipped:`. On this packaged slice (48224 @ 300 ms) both
runs converge at full plasma scale and the TS-only reconstruction tracks the TS+IDS one to
**a few % in p₀ and <1 % in q₀/q₉₅** — consistent with the 45-point validation of the
Ti/Te coefficient (−0.1 % ± 5.6 %), i.e. the statistical approximation is good enough for
routine equilibrium chains, with the real ion data still preferred whenever it exists.

In [ ]:
# --- kinetic EFIT from the TS-only statistical pressure (same base kfile / sweep) ---
efit_result_ts = None
if not EFIT_EXE or not BASE_KFILE:
    print("skipped: set $EFIT and $VAFT_KINETIC_BASE_KFILE (a magnetic kfile) to run "
          "the TS-only kinetic EFIT comparison.")
else:
    try:
        ts_efit_config = KineticEFITConfig(
            executable=EFIT_EXE,
            workdir=str(WORKROOT / "kinetic_efit_ts_only"),
            shot=SHOT,
            time_ms=TIME_MS,
            encoding="raw6",
            base_kfile=BASE_KFILE,      # ti_te_ratio='auto' (default) does the Ti=0.17*Te fallback
            timeout=600,
        )
        ts_inputs = prepare_kinetic_efit_inputs(ods_ts, geq, ts_efit_config)
        efit_result_ts = run_kinetic_efit(ts_inputs, ts_efit_config)
        if efit_result_ts.converged:
            print(f"TS-only kinetic EFIT converged at PLASMA scale = {efit_result_ts.scale}")
        else:
            print(f"TS-only kinetic EFIT did not converge: {efit_result_ts.reason}")
    except Exception as exc:
        print(f"skipped: TS-only kinetic EFIT raised {type(exc).__name__}: {exc}")

# --- side-by-side comparison against the TS+IDS run from section 4 ---
have_ids_run = efit_result is not None and getattr(efit_result, "gfile", None)
have_ts_run  = efit_result_ts is not None and getattr(efit_result_ts, "gfile", None)
if not (have_ids_run and have_ts_run):
    if not have_ids_run:
        print("no TS+IDS kinetic-EFIT solution from section 4 -> nothing to compare against.")
else:
    from vaft.code.kineticEfit import _psin_of_r   # midplane R -> psi_N mapper (display only)

    runs = [
        ("TS + ion Doppler", efit_result,    points, "tab:purple", "-"),
        ("TS-only (Ti=0.17 Te)", efit_result_ts, pts_ts, "tab:orange", "--"),
    ]
    print(f"{'run':<22}{'scale':>7}{'Ip [kA]':>10}{'p0 [Pa]':>10}{'q0':>8}{'q95':>8}")
    scalars = {}
    for label, res, _pts, *_ in runs:
        g = read_geqdsk(str(res.gfile))
        x = np.linspace(0.0, 1.0, int(g["NW"]))
        p = np.asarray(g["PRES"], dtype=float)
        q = np.asarray(g["QPSI"], dtype=float)
        scalars[label] = dict(g=g, x=x, p=p, q=q, ip=float(g["CURRENT"]),
                              p0=p[0], q0=q[0], q95=float(np.interp(0.95, x, q)))
        s = scalars[label]
        print(f"{label:<22}{res.scale:>7.2f}{s['ip']/1e3:>10.1f}{s['p0']:>10.1f}"
              f"{s['q0']:>8.3f}{s['q95']:>8.3f}")
    ref, cmp_ = scalars["TS + ion Doppler"], scalars["TS-only (Ti=0.17 Te)"]
    print(f"{'TS-only vs TS+IDS':<22}{'':>7}"
          f"{(cmp_['ip']/ref['ip']-1)*100:>+9.1f}%{(cmp_['p0']/ref['p0']-1)*100:>+9.1f}%"
          f"{(cmp_['q0']/ref['q0']-1)*100:>+7.1f}%{(cmp_['q95']/ref['q95']-1)*100:>+7.1f}%")

    fig, (ax_p, ax_q, ax_b) = plt.subplots(1, 3, figsize=(16, 5))
    for label, res, _pts, color, ls in runs:
        s = scalars[label]
        ax_p.plot(s["x"], s["p"], color=color, ls=ls, lw=2, label=f"EFIT {label}")
        ax_q.plot(s["x"], s["q"], color=color, ls=ls, lw=2, label=label)
        rb = np.asarray(s["g"].get("RBBBS", []), dtype=float)
        zb = np.asarray(s["g"].get("ZBBBS", []), dtype=float)
        if rb.size and zb.size:
            n = min(rb.size, zb.size)
            ax_b.plot(rb[:n], zb[:n], color=color, ls=ls, lw=2, label=label)
    # overlay each run's constraint points, mapped R -> psi_N through its own solution
    for label, res, _pts, color, _ls in runs:
        rp_i = np.asarray(_pts.rpress, dtype=float)
        pr_i = np.asarray(_pts.pressr, dtype=float)
        sg_i = np.asarray(_pts.sigpre, dtype=float)
        psin_i = np.where(rp_i > 0, _psin_of_r(scalars[label]["g"], rp_i), -rp_i)
        ax_p.errorbar(psin_i, pr_i, yerr=sg_i, fmt="o", color=color, mfc="white",
                      ms=5, capsize=3, elinewidth=1, label=f"points {label}")
    ax_p.set_title("Pressure [Pa]"); ax_p.set_xlabel(r"$\psi_N$")
    ax_p.grid(True, alpha=0.25); ax_p.legend(fontsize=8)
    ax_q.set_title("Safety factor q"); ax_q.set_xlabel(r"$\psi_N$")
    ax_q.grid(True, alpha=0.25); ax_q.legend(fontsize=9)
    ax_b.set_title("Boundary"); ax_b.set_xlabel("R [m]"); ax_b.set_ylabel("Z [m]")
    ax_b.set_aspect("equal", adjustable="box"); ax_b.grid(True, alpha=0.25); ax_b.legend(fontsize=9)
    fig.suptitle(f"Kinetic EFIT: real ion data vs TS-only statistical pressure @ {TIME_MS:.0f} ms",
                 fontsize=13)
    fig.tight_layout()
    plt.show()